# Data Downloader

In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='ccgeo-480116')

### Import asset

In [ ]:
polys = ee.FeatureCollection("projects/ccgeo-480116/assets/SU")

### Collect morphological derivates

In [ ]:
from tagee import terrainAnalysis  # pip install tagee

dem_ic = ee.ImageCollection("COPERNICUS/DEM/GLO30").select("DEM")
dem = dem_ic.filterBounds(polys.geometry()).mosaic()#.clip(polys.geometry())
scale=30

metrics = terrainAnalysis(dem)

# Pick the bands you want (edit as needed)
band_names = [
    "Slope",
    "HorizontalCurvature",
    "VerticalCurvature",
    "MeanCurvature",
    "GaussianCurvature",
    "ShapeIndex",
    "MinimalCurvature",
    "MaximalCurvature"
]
img_morpho = metrics.select(band_names)

### Reducer

In [ ]:
def reducer(image,scale):
  reducer = ee.Reducer.mean().combine(
      reducer2=ee.Reducer.stdDev(),
      sharedInputs=True
  )

  stats_fc = image.reduceRegions(
      collection=polys,
      reducer=reducer,
      scale=scale
  )
  return stats_fc

### Reduce morphological derivates

In [ ]:
stats_morpho= reducer(img_morpho,scale)
print(stats_morpho.first().toDictionary().getInfo())

{'GaussianCurvature_mean': -1.2018913921790634e-06, 'GaussianCurvature_stdDev': 2.4720841393242413e-05, 'HorizontalCurvature_mean': 0.0004071908519356142, 'HorizontalCurvature_stdDev': 0.00528031940377435, 'MaximalCurvature_mean': 0.0033319273841408478, 'MaximalCurvature_stdDev': 0.004424309858998178, 'MeanCurvature_mean': 1.9270864024456877e-05, 'MeanCurvature_stdDev': 0.004085195172595311, 'MinimalCurvature_mean': -0.003295067059417801, 'MinimalCurvature_stdDev': 0.005251631719294957, 'ShapeIndex_mean': 0.07017863377666611, 'ShapeIndex_stdDev': 1.0141279093617024, 'Slope_mean': 26.585244420215783, 'Slope_stdDev': 8.88567061515574, 'VerticalCurvature_mean': -0.0003686491238867009, 'VerticalCurvature_stdDev': 0.004910402328857252, 'id': 1, 'lip': 0}


### Export table to Drive

In [ ]:
task = ee.batch.Export.table.toDrive(
    collection=stats_morpho,
    description="polygons_tagee_stats",
    fileFormat="SHP"
)
task.start()

### Convirt table to Geopandas dataframe

In [ ]:
import geopandas as gdp
import geemap

gdf = geemap.ee_to_gdf(stats_morpho)

### Export dataframe to GeoJSON

In [ ]:
print(gdf.head(10))

                                            geometry  GaussianCurvature_mean  \
0  POLYGON ((12.7952 43.51987, 12.79525 43.51852,...           -1.201891e-06   
1  POLYGON ((12.87085 43.52184, 12.87088 43.52103...           -7.054573e-07   
2  POLYGON ((12.82818 43.52131, 12.82819 43.52104...           -5.792640e-07   
3  POLYGON ((12.87806 43.52792, 12.87813 43.52603...           -7.172605e-07   
4  POLYGON ((12.80134 43.52458, 12.80137 43.52377...           -1.219643e-07   
5  POLYGON ((12.85663 43.52509, 12.85666 43.52428...           -7.794016e-07   
6  POLYGON ((12.83748 43.52095, 12.83752 43.51959...           -5.814081e-07   
7  POLYGON ((12.85055 43.51849, 12.85058 43.51768...           -7.365059e-07   
8  POLYGON ((12.8822 43.52637, 12.88224 43.52529,...           -1.156136e-06   
9  POLYGON ((12.80671 43.51982, 12.80673 43.51928...           -1.051599e-06   

   GaussianCurvature_stdDev  HorizontalCurvature_mean  \
0                  0.000025                  0.000407   
1    

In [ ]:
print(gdf.head(10))
gdf.to_file("/content/drive/MyDrive/CCGEO_folder/morpho.geojson", driver="GeoJSON")

                                            geometry  GaussianCurvature_mean  \
0  POLYGON ((12.7952 43.51987, 12.79525 43.51852,...           -1.201891e-06   
1  POLYGON ((12.87085 43.52184, 12.87088 43.52103...           -7.054573e-07   
2  POLYGON ((12.82818 43.52131, 12.82819 43.52104...           -5.792640e-07   
3  POLYGON ((12.87806 43.52792, 12.87813 43.52603...           -7.172605e-07   
4  POLYGON ((12.80134 43.52458, 12.80137 43.52377...           -1.219643e-07   
5  POLYGON ((12.85663 43.52509, 12.85666 43.52428...           -7.794016e-07   
6  POLYGON ((12.83748 43.52095, 12.83752 43.51959...           -5.814081e-07   
7  POLYGON ((12.85055 43.51849, 12.85058 43.51768...           -7.365059e-07   
8  POLYGON ((12.8822 43.52637, 12.88224 43.52529,...           -1.156136e-06   
9  POLYGON ((12.80671 43.51982, 12.80673 43.51928...           -1.051599e-06   

   GaussianCurvature_stdDev  HorizontalCurvature_mean  \
0                  0.000025                  0.000407   
1    

### Homework
Collect derivatives using GEE catalog, calculate mean and standard deviation per polygon of SU shapefile to one single dataframe or shapefile and export it to your local machine. You should collect:
- DEM derivatives
- mean of temperature in past 10 years (2015-2024)
- mean of yearly cumulated rainfall in past 10 years (2015-2024)
- mean of NDVI in past 10 years (2015-2024)